# Testing => doctest

`doctest` runs the **examples written inside docstrings** and checks that the output matches. It keeps documentation and code in sync.

| Tool | Purpose |
|---|---|
| `>>>` line | An example to run |
| The line(s) after it | The **expected output** |
| `doctest.testmod()` | Test every docstring in the current module |
| `doctest.run_docstring_examples(f, globals())` | Test one function's docstring |
| `python -m doctest file.py -v` | Run from the terminal (verbose) |
| `# doctest: +ELLIPSIS` | Let `...` match any text |
| `# doctest: +NORMALIZE_WHITESPACE` | Ignore differences in spaces and line breaks |
| `# doctest: +SKIP` | Skip an example |

---

## Example

```python
def add(a, b):
    """Return the sum of a and b.

    >>> add(2, 3)
    5
    >>> add(-1, 1)
    0
    """
    return a + b
```

```python
if __name__ == "__main__":
    import doctest
    doctest.testmod()
```

`testmod()` prints nothing when every example passes. Failures show the example, the expected output and the actual output.

---

## Rules

* The output must match **exactly**, including spaces and quotes.
* A blank line ends the expected output. Use `<BLANKLINE>` to expect an empty line.
* Exceptions can be tested by writing the `Traceback` header and the last line:

```python
>>> divide(1, 0)
Traceback (most recent call last):
    ...
ZeroDivisionError: division by zero
```

* Floats, dictionaries with unpredictable order, memory addresses and dates make fragile examples.

---

## Directives

| Directive | Use |
|---|---|
| `+ELLIPSIS` | `<object at ...>` |
| `+NORMALIZE_WHITESPACE` | Long output wrapped over several lines |
| `+SKIP` | An example that cannot run here |

```python
>>> print(list(range(30)))  # doctest: +NORMALIZE_WHITESPACE
[0, 1, 2, ...]
```

---

## `doctest` or `unittest`?

| Need | Use |
|---|---|
| Documentation examples that stay correct | `doctest` |
| Detailed tests, fixtures, many edge cases | `unittest` (or `pytest`) |

Many projects use both.

## Source

https://docs.python.org/3/library/doctest.html

In [ ]:
import contextlib
import doctest
import io

def add(a, b):
    """Return the sum of a and b.

    >>> add(2, 3)
    5
    >>> add(-1, 1)
    0
    """
    return a + b

def divide(a, b):
    """Divide a by b.

    >>> divide(6, 3)
    2.0
    >>> divide(1, 0)
    Traceback (most recent call last):
        ...
    ZeroDivisionError: division by zero
    """
    return a / b

def wrong_double(n):
    """This docstring has a wrong example.

    >>> wrong_double(2)
    5
    """
    return n * 2

def with_ellipsis():
    """The address is different every time, so use ELLIPSIS.

    >>> object()  # doctest: +ELLIPSIS
    <object object at ...>
    """

def run(function):
    """Run the docstring examples of one function and return (failed, attempted)."""
    finder = doctest.DocTestFinder()
    runner = doctest.DocTestRunner(verbose=False)
    for test in finder.find(function, function.__name__, globs=globals()):
        with contextlib.redirect_stdout(io.StringIO()):
            runner.run(test)
    with contextlib.redirect_stdout(io.StringIO()):
        results = runner.summarize(verbose=False)
    return results.failed, results.attempted

print("add:          ", run(add))
print("divide:       ", run(divide))
print("with_ellipsis:", run(with_ellipsis))
print("wrong_double: ", run(wrong_double))          # (1 failed, 1 attempted)

# A failing example shows what was expected and what was received
buffer = io.StringIO()
with contextlib.redirect_stdout(buffer):
    doctest.run_docstring_examples(wrong_double, globals(), name="wrong_double")
report = buffer.getvalue().splitlines()
print([line.strip() for line in report if line.strip() in ("Expected:", "Got:", "5", "4")])

# In a script or a notebook that defines the functions, run every docstring at once:
#     doctest.testmod()